# 🍜 STEP 04 — MCP 연동: 카카오 지도 API / 법령정보 API

**Model Context Protocol(MCP)**을 사용하면 외부 API를 표준 툴로
에이전트에 연결할 수 있습니다.

---

## 이 노트북에서 배울 것

| 개념 | 설명 |
|------|------|
| MCP 개요 | 에이전트 ↔ 외부 시스템 표준 연결 프로토콜 |
| `McpServerPlugin` (SK) | MCP 서버를 SK 플러그인으로 사용 |
| stdio MCP 서버 | 로컬 프로세스로 실행하는 MCP 서버 패턴 |
| API → MCP 래핑 | 카카오 API, 법령 API를 MCP 툴로 변환 |


---
## MCP 개념 이해

```
SK Agent
  └── McpServerPlugin  ←→  MCP Server (stdio/HTTP)
                              ├── 카카오 지도 API 호출
                              ├── 국가법령정보센터 API 호출
                              └── 소상공인 상권정보 API 호출
```

MCP는 LLM 에이전트와 외부 도구 사이의 **표준 통신 프로토콜**입니다.
- **stdio 방식**: 로컬 프로세스로 MCP 서버 실행 (개발·테스트 용이)
- **HTTP 방식**: REST 엔드포인트로 MCP 서버 운영 (프로덕션)

> **F&B 시스템 연결**: LocationAgent의 카카오 지도 API, LegalTaxAgent의 법령 API를 MCP로 연결합니다.

In [ ]:
# MCP 지원을 위한 추가 패키지
%pip install semantic-kernel[mcp] mcp

In [ ]:
import asyncio, os, json
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.functions import KernelArguments

load_dotenv()

def make_kernel():
    k = Kernel()
    k.add_service(AzureChatCompletion(
        deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    ))
    return k

def make_auto_settings():
    s = AzureChatPromptExecutionSettings()
    s.function_choice_behavior = FunctionChoiceBehavior.Auto()
    return s

print("임포트 완료")

---
## 1. MCP 서버 직접 구현 — 카카오 지도 API 래핑

> 아래 코드를 `kakao_mcp_server.py`로 저장하면 독립적인 MCP 서버가 됩니다.
> SK 에이전트는 이 서버를 `McpServerPlugin`으로 연결합니다.

In [ ]:
# ============================================================
# kakao_mcp_server.py — 별도 파일로 저장 후 실행
# 실행: python kakao_mcp_server.py
# ============================================================
KAKAO_MCP_SERVER_CODE = '''
import asyncio
import os
import httpx
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent

# MCP 서버 인스턴스
server = Server("kakao-location-server")

KAKAO_API_KEY = os.getenv("KAKAO_REST_API_KEY", "")
KAKAO_LOCAL_BASE = "https://dapi.kakao.com/v2/local"


@server.list_tools()
async def list_tools() -> list[Tool]:
    """이 MCP 서버가 제공하는 툴 목록을 반환합니다."""
    return [
        Tool(
            name="search_places",
            description="카카오 지도 API로 특정 위치 근처의 장소를 검색합니다.",
            inputSchema={
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "검색어 (예: 마포구 홍대 카페)"},
                    "radius_m": {"type": "integer", "description": "검색 반경 (미터, 기본 500)"},
                    "size": {"type": "integer", "description": "결과 개수 (기본 10)"},
                },
                "required": ["query"],
            },
        ),
        Tool(
            name="get_address_info",
            description="주소 또는 키워드로 위치 좌표와 행정구역 정보를 조회합니다.",
            inputSchema={
                "type": "object",
                "properties": {
                    "address": {"type": "string", "description": "주소 또는 장소명"},
                },
                "required": ["address"],
            },
        ),
    ]


@server.call_tool()
async def call_tool(name: str, arguments: dict) -> list[TextContent]:
    """툴 호출을 처리합니다."""
    headers = {"Authorization": "KakaoAK %s" % KAKAO_API_KEY}

    async with httpx.AsyncClient() as client:
        if name == "search_places":
            # 카카오 키워드 검색 API
            resp = await client.get(
                "%s/search/keyword.json" % KAKAO_LOCAL_BASE,
                headers=headers,
                params={
                    "query": arguments["query"],
                    "radius": arguments.get("radius_m", 500),
                    "size": arguments.get("size", 10),
                },
            )
            data = resp.json()
            places = data.get("documents", [])
            result = [{
                "name": p["place_name"],
                "category": p["category_name"],
                "address": p["address_name"],
                "distance_m": p.get("distance", ""),
            } for p in places]
            return [TextContent(type="text", text=str(result))]

        elif name == "get_address_info":
            # 카카오 주소 검색 API
            resp = await client.get(
                "%s/search/address.json" % KAKAO_LOCAL_BASE,
                headers=headers,
                params={"query": arguments["address"]},
            )
            data = resp.json()
            return [TextContent(type="text", text=str(data.get("documents", [])))]

    return [TextContent(type="text", text="알 수 없는 툴: %s" % name)]


async def main():
    async with stdio_server() as (read, write):
        await server.run(read, write, server.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())
'''

# 파일로 저장
with open("kakao_mcp_server.py", "w", encoding="utf-8") as f:
    f.write(KAKAO_MCP_SERVER_CODE)

print("kakao_mcp_server.py 저장 완료!")
print("실행: python kakao_mcp_server.py")

---
## 2. SK에서 MCP 서버 연결 — `McpServerPlugin`

In [ ]:
import sys
from semantic_kernel.connectors.mcp import McpServerPlugin, MpcStdioServerParams

async def create_location_agent_with_mcp() -> ChatCompletionAgent:
    """
    카카오 지도 MCP 서버를 플러그인으로 연결한 LocationAgent를 생성합니다.
    
    동작 흐름:
    1. McpServerPlugin이 kakao_mcp_server.py를 서브프로세스로 실행
    2. stdio로 MCP 프로토콜 통신
    3. 서버가 제공하는 툴이 SK 플러그인 함수로 자동 등록
    """
    kernel = make_kernel()

    # MCP 서버를 플러그인으로 연결
    # MpcStdioServerParams: stdio 방식으로 MCP 서버 프로세스를 실행
    mcp_plugin = await McpServerPlugin.create(
        MpcStdioServerParams(
            command=sys.executable,        # python 실행 경로
            args=["kakao_mcp_server.py"],  # MCP 서버 스크립트
            env={
                "KAKAO_REST_API_KEY": os.getenv("KAKAO_REST_API_KEY", ""),
            },
        ),
        plugin_name="KakaoLocation",
    )
    kernel.add_plugin(mcp_plugin)

    return ChatCompletionAgent(
        name="LocationAgent",
        description="카카오 지도 API 기반 상권 분석 에이전트",
        instructions=(
            "당신은 상권 분석 전문가입니다.\n"
            "KakaoLocation 플러그인의 search_places 툴로 실제 장소 데이터를 조회하여 분석합니다.\n"
            "경쟁업체 수, 업종 분포, 접근성을 구체적 수치와 함께 제공하세요."
        ),
        kernel=kernel,
        arguments=KernelArguments(settings=make_auto_settings()),
    )

print("MCP 연동 LocationAgent 팩토리 함수 정의 완료")
print()
print("⚠️ 실제 실행 전 준비사항:")
print("  1. .env에 KAKAO_REST_API_KEY 설정")
print("  2. pip install mcp httpx")
print("  3. kakao_mcp_server.py가 같은 폴더에 있어야 함")

In [ ]:
# 카카오 API 키가 있을 때만 실행
KAKAO_KEY = os.getenv("KAKAO_REST_API_KEY", "")

if KAKAO_KEY:
    location_agent_mcp = await create_location_agent_with_mcp()
    response = await location_agent_mcp.get_response(
        messages="서울 마포구 홍대입구역 근처 카페가 얼마나 있어? 반경 500m 내로 조회해줘."
    )
    print(response.content)
else:
    print("KAKAO_REST_API_KEY가 설정되지 않았습니다.")
    print(".env 파일에 KAKAO_REST_API_KEY=xxxxxxx 를 추가하세요.")

---
## 3. 국가법령정보센터 API MCP 서버

> LegalTaxAgent에 연결할 법령 API MCP 서버 코드입니다.

In [ ]:
LAW_MCP_SERVER_CODE = '''
import asyncio
import os
import httpx
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent

server = Server("law-info-server")

LAW_API_KEY = os.getenv("LAW_API_KEY", "")  # 국가법령정보센터 발급 키
LAW_BASE = "https://www.law.go.kr/DRF/lawSearch.do"


@server.list_tools()
async def list_tools() -> list[Tool]:
    return [
        Tool(
            name="search_law",
            description="국가법령정보센터에서 법령을 검색합니다. 식품위생법, 상가임대차보호법 등 F&B 관련 법령 조회.",
            inputSchema={
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "검색할 법령명 또는 키워드"},
                },
                "required": ["query"],
            },
        ),
        Tool(
            name="get_law_article",
            description="특정 법령의 조문 내용을 조회합니다.",
            inputSchema={
                "type": "object",
                "properties": {
                    "law_name": {"type": "string", "description": "법령명 (예: 식품위생법)"},
                    "article_no": {"type": "string", "description": "조문 번호 (예: 제37조)"},
                },
                "required": ["law_name", "article_no"],
            },
        ),
    ]


@server.call_tool()
async def call_tool(name: str, arguments: dict) -> list[TextContent]:
    async with httpx.AsyncClient() as client:
        if name == "search_law":
            resp = await client.get(
                LAW_BASE,
                params={
                    "OC": LAW_API_KEY,
                    "target": "law",
                    "type": "JSON",
                    "query": arguments["query"],
                    "display": 5,
                },
            )
            return [TextContent(type="text", text=resp.text)]

    return [TextContent(type="text", text="알 수 없는 툴: %s" % name)]


async def main():
    async with stdio_server() as (read, write):
        await server.run(read, write, server.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())
'''

with open("law_mcp_server.py", "w", encoding="utf-8") as f:
    f.write(LAW_MCP_SERVER_CODE)

print("law_mcp_server.py 저장 완료!")